[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/motor_loop/motor_loop.ipynb)

# Loop Subdivision of Motors

This example computes Loop subdivision on a triangulated surface whose vertices carry rigid placements of frames rather than positions alone. Each Loop step refines the surface by averaging vertex values across the topology with fixed stencil weights determined solely by connectivity. Winding up a ribbon by an axial twist contrasts position averaging against placement averaging: subdividing positions wrings the middle of the ribbon thin because the average of two rotated positions cuts straight across the chord of the turn, whereas subdividing rigid placements keeps the full width because normalizing the averaged placement places the refined frame out on the arc halfway around the turn.

Each vertex placement is modelled as a motor, and each Loop step is a linear map from coarse vertex sites to fine vertex sites that acts directly via `step * motors`. Because a weighted sum of motors leaves the group, normalizing at the end of every refinement level with `(step * motors).normalized()` returns each averaged element to a unit motor before the next step, while sandwiching the origin `motors >> origin` reads out the refined surface.

In matrix and dual-quaternion subdivision notation, linear point refinement multiplies a coarse-to-fine stencil matrix $S$ into the stacked vertex coordinates $P' = S P$, whereas rigid-frame refinement represents each pose $(R_i, \mathbf t_i) \in \mathrm{SE}(3)$ as a unit dual quaternion $\hat q_i = q_{r,i} + \tfrac{1}{2}\varepsilon\,(0, \mathbf t_i)\,q_{r,i}$, blends rows linearly via $\tilde q_j = \sum_i S_{ji}\,\hat q_i$ with hemisphere sign alignment when needed, normalizes each blended dual quaternion by $\|\tilde q_{r,j}\|$, and recovers vertex positions from $2\,\hat q_{\varepsilon,j}\,\hat q_{r,j}^*$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.topology import SubdividedTopology, Topology
from examples.util.animation import show_states
from examples.surfaces.motor_loop import render, scenarios

ctx = NumpyContext(ga)
mv = ctx.multivector
Line = ga.gatype.bivector()                                                 # Bivector+
Point = ga.gatype.antivector()                                              # Antivector-
Motor = ga.gatype.rotor()                                                   # Rotor+
origin = mv.w.dual()                                                        # Point-

## Subdividing motors and points on a triangulated surface

A Loop refinement step depends on triangle incidences alone, mapping vertex values on the coarse topology linearly onto the refined vertices and edge midpoints while holding pinned corners fixed. Point subdivision needs no intermediate projection, so its refinement steps compose into one operator `topology.subdivision`. Motor subdivision normalizes `(step * motors).normalized()` after each level so every refined site carries a genuine rigid motion before the next stencil averages it.

In [ ]:
control_topology, refined_topology, z_coords, x_coords = scenarios.ribbon_setup()
axis_length = scenarios.axis_length

In [ ]:
def subdivided_motors(topology: SubdividedTopology, motors: Motor) -> Motor:
    """Repeated Loop refinement averages `motors` across each level of `topology` and normalizes every sum back to a unit motor (`[...] Motor+[V_refined]`)."""
    for step in topology.steps:
        motors = (step * motors).normalized()                               # [...] Motor+[V_finer]
    return motors


def frame_markers(axis_length: float) -> Point:
    """Four reference points mark a frame's origin followed by the tips of its three axes at `axis_length` (`[4] Point-`)."""
    return stack([
        origin,
        mv.point([axis_length, 0.0, 0.0]),
        mv.point([0.0, axis_length, 0.0]),
        mv.point([0.0, 0.0, axis_length]),
    ])                                                                      # [4] Point-

In [ ]:
def subdivided_surfaces(topology: SubdividedTopology, motors: Motor) -> Point:
    """Point subdivision refines the vertex origins directly while motor subdivision refines and normalizes `motors` at each level (`[..., 2] Point-[V_refined]`)."""
    point_surface = topology.subdivision * (motors >> origin)               # [...] Point-[V_refined]
    motor_surface = subdivided_motors(topology, motors) >> origin           # [...] Point-[V_refined]
    return stack([point_surface, motor_surface], axis=-1)                   # [..., 2] Point-[V_refined]


def ribbon_rest_motors(z_coords: np.ndarray, x_coords: np.ndarray) -> Motor:
    """Translations along `z_coords` and across `x_coords` place flat vertex frames across a `[V]` ribbon grid (`[V] Motor+`)."""
    return (mv.zw * (z_coords / 2)).exp() * (mv.xw * (x_coords / 2)).exp()  # [V] Motor+


markers = frame_markers(axis_length)                                        # [4] Point-
rest_motors = ribbon_rest_motors(z_coords, x_coords)                        # [V] Motor+

## Winding up the ribbon

Holding the near end fixed while winding the far end one and a half turns and releasing it grows the twist angle quadratically along the ribbon. Point subdivision averages positions across the turn and pinches the waist inward toward the central axis, whereas motor subdivision averages and normalizes vertex motors at each level so every refined frame stays out at its full radial distance.

In dual-quaternion skinning notation, normalizing a linear combination of unit dual quaternions corresponds to dual-quaternion linear blending, whereas averaging point coordinates directly exhibits the candy-wrapper volume collapse of linear blend skinning.

In [ ]:
length = scenarios.length
twist_angle = scenarios.twist_angle
n_frames = scenarios.n_frames
duration_ms = scenarios.duration_ms
winding_view = render.ribbon_view(scenarios.winding_bounds(rest_motors, markers, z_coords))

In [ ]:
def ribbon_states(topology: SubdividedTopology, markers: Point, motors: Motor) -> Iterator[tuple[Point, Point]]:
    """Yields `(control_frames, surfaces)` (`[4] Point-[V]`, `[2] Point-[V_refined]`) for each frame of `motors`."""
    for frame_motors in motors:
        yield frame_motors[None] >> markers, subdivided_surfaces(topology, frame_motors)


def winding_states(topology: SubdividedTopology, markers: Point, rest_motors: Motor,
                   z_coords: np.ndarray, length: float, twist_angle: float, n_frames: int) -> Iterator[tuple[Point, Point]]:
    """Yields `(control_frames, surfaces)` across `n_frames` as the ribbon winds up to `twist_angle` and unwinds back to rest."""
    twist_angles = twist_angle * (1 - np.cos(np.linspace(0, 2 * np.pi, n_frames, endpoint=False))) / 2  # [n_frames]
    twist_profile = ((z_coords + length / 2) / length) ** 2                 # [V]
    twist_turns = (mv.xy * (-twist_angles[:, None] * twist_profile / 2)).exp()  # [n_frames, V] Motor+
    motors = (twist_turns * rest_motors).field()                            # [n_frames] Motor+[V]
    yield from ribbon_states(topology, markers, motors)


recorded_winding = show_states(
    winding_states(refined_topology, markers, rest_motors, z_coords, length, twist_angle, n_frames),
    render.surface_animation, control_topology, refined_topology, winding_view,
    name="motor_loop_ribbon", duration_ms=duration_ms, n_frames=n_frames,
);

Point subdivision is shown on the left and motor subdivision on the right, with red, green, and blue arrows marking the local frame axes at each vertex of the dashed control net.

## Turning vertex frames in place

When every control point stays fixed on the flat ribbon while each vertex frame oscillates around a line through its own origin at an independent phase, the point-subdivided surface remains completely flat. The motor-subdivided surface bows out of the plane because the normalized average of two differently oriented frames follows the screw motion between them. Right-multiplying every vertex motor by a single shared local rotor leaves the subdivided surface unchanged because the common right factor cancels when `motors >> origin` acts on the origin; only relative turns between neighbouring frames bend the surface.

In [ ]:
turn_amplitude = scenarios.turn_amplitude
axes, phases = scenarios.turning_axes(control_topology)
turning_view = render.ribbon_view(scenarios.turning_bounds(rest_motors, markers, axes))

In [ ]:
def turning_states(topology: SubdividedTopology, markers: Point, rest_motors: Motor,
                   axes: Line, phases: np.ndarray, turn_amplitude: float, n_frames: int) -> Iterator[tuple[Point, Point]]:
    """Yields `(control_frames, surfaces)` across `n_frames` as vertex frames oscillate by `turn_amplitude` around `axes` at `phases`."""
    angles = turn_amplitude * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None] + phases)  # [n_frames, V]
    local_turns = (axes * (-angles / 2)).exp()                              # [n_frames, V] Motor+
    motors = (rest_motors * local_turns).field()                            # [n_frames] Motor+[V]
    yield from ribbon_states(topology, markers, motors)


recorded_turning = show_states(
    turning_states(refined_topology, markers, rest_motors, axes, phases, turn_amplitude, n_frames),
    render.surface_animation, control_topology, refined_topology, turning_view,
    name="motor_loop_turning", duration_ms=duration_ms, n_frames=n_frames,
);

Even with stationary vertex origins on the dashed control net, opposing local frame rotations bow the motor-subdivided sheet on the right while leaving the point-subdivided sheet on the left motionless.

In [ ]:
# checks
# Untwisted, both schemes agree; wound up, point subdivision wrings the ribbon inward while motor subdivision keeps its width.
centre_line = origin & mv.point([0.0, 0.0, 1.0])                            # [] Antibivector-
winding_endpoints = stack([recorded_winding[0][1], recorded_winding[n_frames // 2][1]]).batch()  # [2, 2, V_refined] Point-
radial_distance = (centre_line & winding_endpoints).norm().mean(axis=-1)    # [2, 2] Scalar+
(points_rest, motors_rest), (points_peak, motors_peak) = radial_distance
assert (points_rest - motors_rest).abs() < 1e-12
assert points_peak < 0.7 * points_rest and motors_peak >= motors_rest

# Turning frames in place leaves the point-subdivided surface fixed while bowing the motor-subdivided surface.
turning_pair = stack([state[1] for state in recorded_turning]).batch()      # [n_frames, 2, V_refined] Point-
point_surfaces, motor_surfaces = turning_pair[:, 0], turning_pair[:, 1]     # [n_frames, V_refined] Point-, [n_frames, V_refined] Point-
assert ((point_surfaces & point_surfaces[:1]).norm() < 1e-12).all()
assert ((motor_surfaces & point_surfaces).norm() > 0.05).any()

# Right-multiplying every vertex motor by a common local rotation leaves the subdivided motor surface unchanged.
shared_turn = (mv.yz * (-0.7 / 2)).exp()                                    # [] Motor+
shared_pair = (subdivided_motors(refined_topology, stack([rest_motors, rest_motors * shared_turn]).field()) >> origin).batch()  # [2, V_refined] Point-
assert ((shared_pair[0] & shared_pair[1]).norm() < 1e-12).all()